In [2]:
# content_based_ecommerce.py

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------
# STEP 1: CREATE DATASET (INLINE)
# -------------------------------
data = {
    "product_id": [1,2,3,4,5,6,7,8,9,10],
    "name": [
        "iPhone 14", "Samsung Galaxy S23", "Redmi Note 12",
        "MacBook Air", "Dell XPS 13", "HP Pavilion",
        "Sony Headphones", "Boat Earbuds",
        "Apple Watch", "Fitbit Charge"
    ],
    "category": [
        "Mobile","Mobile","Mobile",
        "Laptop","Laptop","Laptop",
        "Accessories","Accessories","Accessories","Accessories"
    ],
    "brand": [
        "Apple","Samsung","Xiaomi",
        "Apple","Dell","HP",
        "Sony","Boat","Apple","Fitbit"
    ],
    "description": [
        "Smartphone with A15 chip and excellent camera",
        "Android phone with powerful processor and camera",
        "Budget phone with long battery life",
        "Lightweight laptop with M1 chip",
        "Premium ultrabook with sleek design",
        "Affordable laptop for students",
        "Noise cancelling over ear headphones",
        "Wireless earbuds with deep bass",
        "Smartwatch with fitness tracking features",
        "Fitness tracker with heart rate monitor"
    ]
}

df = pd.DataFrame(data)

# -------------------------------
# STEP 2: PREPROCESSING
# -------------------------------
# Convert to lowercase for consistency
for col in ['name', 'category', 'brand', 'description']:
    df[col] = df[col].str.lower()

# Combine features
df['features'] = (
    df['name'] + " " +
    df['category'] + " " +
    df['brand'] + " " +
    df['description']
)

# -------------------------------
# STEP 3: TF-IDF VECTORIZATION
# -------------------------------
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['features'])

# -------------------------------
# STEP 4: COSINE SIMILARITY
# -------------------------------
similarity_matrix = cosine_similarity(tfidf_matrix)

# -------------------------------
# STEP 5: RECOMMENDATION FUNCTION
# -------------------------------
def recommend(product_name, top_n=5):
    product_name = product_name.lower()
    
    # Check if product exists
    if product_name not in df['name'].values:
        return ["❌ Product not found"]
    
    # Get index
    index = df[df['name'] == product_name].index[0]
    
    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[index]))
    
    # Sort by score (highest first)
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    # Get top N (skip itself)
    top_products = similarity_scores[1:top_n+1]
    
    results = []
    for i in top_products:
        product = df.iloc[i[0]]
        results.append({
            "name": product['name'].title(),
            "category": product['category'].title(),
            "score": round(i[1], 3)
        })
    
    return results

# -------------------------------
# STEP 6: TEST THE MODEL
# -------------------------------
if __name__ == "__main__":
    product = "iPhone 14"
    
    print(f"\n🔍 Recommendations for: {product}\n")
    
    recommendations = recommend(product)
    
    for i, item in enumerate(recommendations, start=1):
        print(f"{i}. {item['name']} ({item['category']}) - Score: {item['score']}")
        


🔍 Recommendations for: iPhone 14

1. Macbook Air (Laptop) - Score: 0.17
2. Apple Watch (Accessories) - Score: 0.148
3. Samsung Galaxy S23 (Mobile) - Score: 0.14
4. Redmi Note 12 (Mobile) - Score: 0.066
5. Dell Xps 13 (Laptop) - Score: 0.0


In [ ]:
import os
import zipfile
from PIL import Image
from tqdm import tqdm
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

# -------------------------------
# CHECK TORCH
# -------------------------------
print("Torch Version:", torch.__version__)

# -------------------------------
# PATHS
# -------------------------------
ZIP_PATH = r"D:\New folder\Object Detection-Images.zip"
EXTRACT_PATH = r"D:\New folder\images"

# -------------------------------
# STEP 1: EXTRACT ZIP (SAFE)
# -------------------------------
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"ZIP file not found: {ZIP_PATH}")

if not os.path.exists(EXTRACT_PATH) or len(os.listdir(EXTRACT_PATH)) == 0:
    print("Extracting ZIP...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("ZIP Extracted!")
else:
    print("ZIP already extracted.")

# -------------------------------
# STEP 2: LOAD MODEL
# -------------------------------
print("Loading AI model...")

try:
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
except Exception as e:
    print("❌ Error loading model:", e)
    print("👉 Check internet connection or install transformers properly")
    exit()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()  # important for inference

print(f"Model Loaded on {device}")

# -------------------------------
# STEP 3: GET IMAGE FILES
# -------------------------------
image_files = []

for root, dirs, files in os.walk(EXTRACT_PATH):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_files.append(os.path.join(root, file))

if len(image_files) == 0:
    raise ValueError("❌ No images found! Check ZIP contents.")

print(f"✅ Found {len(image_files)} images")

# -------------------------------
# STEP 4: GENERATE CAPTION
# -------------------------------
def generate_caption(image_path):
    try:
        with Image.open(image_path) as image:
            image = image.convert('RGB')

            inputs = processor(images=image, return_tensors="pt").to(device)

            with torch.no_grad():
                output = model.generate(**inputs, max_length=50)

            caption = processor.decode(output[0], skip_special_tokens=True)
            return caption

    except Exception as e:
        return f"Error: {e}"

# -------------------------------
# STEP 5: RUN
# -------------------------------
print("\nGenerating captions...\n")

for img_path in tqdm(image_files[:10]):  # limit to 10
    caption = generate_caption(img_path)

    print(f"\n🖼 Image: {os.path.basename(img_path)}")
    print(f"📝 Caption: {caption}")
    print("-" * 50)